In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/interim/registros_academicos_raw.csv"
)

df.head()

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante,programa_estudios
0,2024-I,I,DESARROLLO PERSONAL I,LOGRADO,15,3,45,4041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
1,2024-I,I,DESARROLLO Y APRENDIZAJE I,EN PROCESO,13,5,65,4041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
2,2024-I,I,FUNDAMENTOS PARA LA EDUCACIÓN PRIMARIA,LOGRADO,15,5,75,4041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
3,2024-I,I,LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR,LOGRADO,15,3,45,4041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
4,2024-I,I,PRÁCTICA E INVESTIGACIÓN I,EN PROCESO,13,5,65,4041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12546 entries, 0 to 12545
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   periodo            12546 non-null  str  
 1   ciclo              12546 non-null  str  
 2   curso              12546 non-null  str  
 3   estado             12546 non-null  str  
 4   nota               12546 non-null  int64
 5   credito            12546 non-null  int64
 6   puntaje            12546 non-null  int64
 7   id_estudiante      12546 non-null  int64
 8   programa_estudios  12546 non-null  str  
dtypes: int64(4), str(5)
memory usage: 882.3 KB


In [3]:
df["id_estudiante"].nunique()

244

In [4]:
trayectorias = df.groupby("id_estudiante").agg(
    periodos_registrados=("periodo", "nunique")
).reset_index()

trayectorias.head()

,id_estudiante,periodos_registrados
0,4041859,5
1,10610527,5
2,20107537,5
3,20893955,2
4,21063279,8


In [5]:
promedio = df.groupby("id_estudiante").agg(
    promedio_global=("nota", "mean")
).reset_index()

promedio.head()

,id_estudiante,promedio_global
0,4041859,14.545455
1,10610527,12.086207
2,20107537,12.228070
3,20893955,5.583333
4,21063279,13.688312


In [6]:
creditos = df.groupby("id_estudiante").agg(
    creditos_totales=("credito", "sum")
).reset_index()

In [7]:
rendimiento = df.groupby("id_estudiante").agg(
    cursos_aprobados=("nota", lambda x: (x >= 11).sum()),
    cursos_desaprobados=("nota", lambda x: (x < 11).sum())
).reset_index()

In [8]:
previo_inicio = (
    df.groupby("id_estudiante")
    .agg(
        cursos_previo_inicio=(
            "estado",
            lambda x: (x == "PREVIO AL INICIO").sum()
        )
    )
    .reset_index()
)

In [9]:

promedio_por_periodo = (
    df.groupby(["id_estudiante", "periodo"], as_index=False)
    .agg(promedio_periodo=("nota", "mean"))
    .sort_values(["id_estudiante", "periodo"])
)

tendencia = (
    promedio_por_periodo
    .groupby("id_estudiante")
    .agg(
        promedio_inicial=("promedio_periodo", "first"),
        promedio_final=("promedio_periodo", "last"),
        periodos_validos=("periodo", "nunique")
    )
    .reset_index()
)

tendencia = tendencia[
    tendencia["periodos_validos"] >= 2
].drop(columns=["periodos_validos"])

tendencia["tendencia"] = (
    tendencia["promedio_final"]
    - tendencia["promedio_inicial"]
)

print("Trayectorias válidas:", len(tendencia))
tendencia.head()


Trayectorias válidas: 244


,id_estudiante,promedio_inicial,promedio_final,tendencia
0,4041859,14.333333,14.428571,0.095238
1,10610527,14.833333,0.000000,-14.833333
2,20107537,14.583333,0.000000,-14.583333
3,20893955,10.166667,1.000000,-9.166667
4,21063279,12.692308,16.500000,3.807692


In [10]:
df_trayectoria = (
    trayectorias
    .merge(promedio, on="id_estudiante")
    .merge(creditos, on="id_estudiante")
    .merge(rendimiento, on="id_estudiante")
    .merge(previo_inicio, on="id_estudiante")
    .merge(tendencia, on="id_estudiante")
)
df_trayectoria.head()

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia
0,4041859,5,14.545455,117,33,0,0,14.333333,14.428571,0.095238
1,10610527,5,12.086207,111,48,10,10,14.833333,0.000000,-14.833333
2,20107537,5,12.228070,111,48,9,9,14.583333,0.000000,-14.583333
3,20893955,2,5.583333,48,5,7,7,10.166667,1.000000,-9.166667
4,21063279,8,13.688312,184,76,1,0,12.692308,16.500000,3.807692


In [11]:
df_trayectoria.shape

(244, 10)

In [12]:
df_trayectoria.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_estudiante         244 non-null    int64  
 1   periodos_registrados  244 non-null    int64  
 2   promedio_global       244 non-null    float64
 3   creditos_totales      244 non-null    int64  
 4   cursos_aprobados      244 non-null    int64  
 5   cursos_desaprobados   244 non-null    int64  
 6   cursos_previo_inicio  244 non-null    int64  
 7   promedio_inicial      244 non-null    float64
 8   promedio_final        244 non-null    float64
 9   tendencia             244 non-null    float64
dtypes: float64(4), int64(6)
memory usage: 19.2 KB


In [13]:
df_trayectoria.describe()

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia
count,2.440000e+02,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000
mean,6.194711e+07,6.573770,13.190643,145.135246,47.118852,4.299180,3.110656,14.022755,11.515165,-2.507590
std,1.680977e+07,3.236668,2.628901,70.126415,30.453469,6.058412,5.224984,2.116778,6.004777,6.021571
min,4.041859e+06,2.000000,4.083333,41.000000,4.000000,0.000000,0.000000,0.000000,0.000000,-15.500000
25%,4.694427e+07,4.000000,11.790552,71.000000,19.750000,0.000000,0.000000,13.666667,9.229167,-3.791667
50%,7.145250e+07,7.000000,14.078030,151.000000,44.500000,1.000000,0.000000,14.333333,14.309524,-0.258333
75%,7.486244e+07,10.000000,15.051890,219.250000,76.500000,7.000000,6.000000,15.020833,15.666667,1.000000
max,7.854754e+07,12.000000,16.612245,270.000000,110.000000,26.000000,25.000000,17.000000,18.500000,13.000000


In [14]:
(df_trayectoria["promedio_inicial"] == 0).sum()

np.int64(1)

In [15]:
(df_trayectoria["promedio_final"] == 0).sum()

np.int64(27)

In [16]:
df_trayectoria["tasa_aprobacion"] = (
    df_trayectoria["cursos_aprobados"] /
    (
        df_trayectoria["cursos_aprobados"] +
        df_trayectoria["cursos_desaprobados"]
    )
)

In [17]:
df_trayectoria.head()

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion
0,4041859,5,14.545455,117,33,0,0,14.333333,14.428571,0.095238,1.000000
1,10610527,5,12.086207,111,48,10,10,14.833333,0.000000,-14.833333,0.827586
2,20107537,5,12.228070,111,48,9,9,14.583333,0.000000,-14.583333,0.842105
3,20893955,2,5.583333,48,5,7,7,10.166667,1.000000,-9.166667,0.416667
4,21063279,8,13.688312,184,76,1,0,12.692308,16.500000,3.807692,0.987013


In [18]:
df_trayectoria.describe()

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion
count,2.440000e+02,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000,244.000000
mean,6.194711e+07,6.573770,13.190643,145.135246,47.118852,4.299180,3.110656,14.022755,11.515165,-2.507590,0.881294
std,1.680977e+07,3.236668,2.628901,70.126415,30.453469,6.058412,5.224984,2.116778,6.004777,6.021571,0.176587
min,4.041859e+06,2.000000,4.083333,41.000000,4.000000,0.000000,0.000000,0.000000,0.000000,-15.500000,0.241379
25%,4.694427e+07,4.000000,11.790552,71.000000,19.750000,0.000000,0.000000,13.666667,9.229167,-3.791667,0.806989
50%,7.145250e+07,7.000000,14.078030,151.000000,44.500000,1.000000,0.000000,14.333333,14.309524,-0.258333,0.983609
75%,7.486244e+07,10.000000,15.051890,219.250000,76.500000,7.000000,6.000000,15.020833,15.666667,1.000000,1.000000
max,7.854754e+07,12.000000,16.612245,270.000000,110.000000,26.000000,25.000000,17.000000,18.500000,13.000000,1.000000


In [19]:
variabilidad = (
    df.groupby("id_estudiante")
    .agg(
        variabilidad_nota=("nota", "std")
    )
    .reset_index()
)

In [20]:
df_trayectoria = (
    df_trayectoria
    .merge(
        variabilidad,
        on="id_estudiante"
    )
)

In [21]:
df_trayectoria.to_csv(
    "../data/processed/trayectorias_academicas.csv",
    index=False,
    encoding="utf-8-sig"
)

In [23]:

import pandas as pd

df_guardado = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

print("Archivo guardado correctamente")
print("Dimensiones:", df_guardado.shape)
print("Estudiantes únicos:", df_guardado["id_estudiante"].nunique())
print("Valores faltantes:", df_guardado.isnull().sum().sum())


Archivo guardado correctamente
Dimensiones: (244, 12)
Estudiantes únicos: 244
Valores faltantes: 0


In [22]:
print("Dimensiones:", df_trayectoria.shape)
print("Estudiantes únicos:", df_trayectoria["id_estudiante"].nunique())
print("Valores faltantes:")
print(df_trayectoria.isnull().sum())

Dimensiones: (244, 12)
Estudiantes únicos: 244
Valores faltantes:
id_estudiante           0
periodos_registrados    0
promedio_global         0
creditos_totales        0
cursos_aprobados        0
cursos_desaprobados     0
cursos_previo_inicio    0
promedio_inicial        0
promedio_final          0
tendencia               0
tasa_aprobacion         0
variabilidad_nota       0
dtype: int64
